In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# use gaussian random walk to extract non linear trends (booming or halting economiy) as opposed to rigid traditional LR

t_months = np.arange(36)
true_trend = 0.5*t_months # steadily increasing
demand = true_trend + np.random.normal(0,2, 36)


In [ ]:
with pm.Model() as bsts_trend: #bayesian_stat_time_serie

    sigma_trend = pm.HalfNormal("sigma_trend", sigma=0.5)
    # how wildly baseline economy can random jump

    trend_component = pm.GaussianRandomWalk("trend_component", sigma=sigma_trend, shape=36)

    sigma_obs = pm.HalfNormal("sigma_obs", sigma=5)
    y = pm.Normal("y", mu=trend_component, sigma=sigma_obs, observed=demand)


    trace_trend = pm.sample(draws=2000, chains=2, cores=2, target_accept=0.95, progressbar=False)
    # complex grw geometry

    prior_checks = pm.sample_prior_predictive(draws=1000, random_seed=42)
    posterior_checks = pm.sample_posterior_predictive(trace=trace_trend, random_seed=42, progressbar=False)

In [ ]:

# The GRW diverges wildly in the prior. We constrain the X-axis to see the core probability mass.
az.plot_ppc(prior_checks, group="prior")
plt.xlim(-50, 100)
plt.title("Prior Predictive Check: Unconstrained Beliefs")
plt.tight_layout()
plt.show()

# Only plot scalar parameters for convergence. Plotting all 36 GRW dimensions is a visual liability.
az.plot_trace(trace_trend, var_names=["sigma_trend", "sigma_obs"])
plt.suptitle("Convergence Check: Trace plots (Scalars only)")
plt.tight_layout()
plt.show()

# Summarize only the core drivers, not every step of the random walk
print(az.summary(trace_trend, var_names=["sigma_trend", "sigma_obs"]))

az.plot_ppc(posterior_checks)
plt.title("Posterior Predictive Check: Model vs Reality")
plt.tight_layout()
plt.show()

In [ ]:
post_trend = trace_trend.posterior["trend_component"].mean(dim=["chain", "draw"])
plt.plot(t_months, demand, "k.", label="Raw Demand Data")
plt.plot(t_months, post_trend, "r-", label="Extracted Trend (GRW)", linewidth=2)

# Always visualize the HDI to show the model's confidence in the recovered trend
hdi = az.hdi(trace_trend.posterior["trend_component"], hdi_prob=0.95)["trend_component"]
plt.fill_between(t_months, hdi[:, 0], hdi[:, 1], color="red", alpha=0.2, label="95% HDI")

plt.legend()
plt.title("Commercial Liability: Capturing the Baseline Economic Growth")
plt.tight_layout()
plt.show()

In [ ]:
# The Black Swan
true_trend[18:24] += 15 # boom
true_trend[24:] -= 10 # followed by crash
demand = true_trend + np.random.normal(0, 2, 36) # rerun observation


with pm.Model() as reg_model:
    # define broad priors for them.
    intercept = pm.Normal("intercept", mu=0, sigma=10)
    slope = pm.Normal("slope", mu=0, sigma=2)
    sigma = pm.HalfNormal("sigma", sigma=5)


    exp_trend = pm.Deterministic("exp_trend", intercept + slope * t_months)

    # The Likelihood: Grounding the rigid line to our chaotic demand data
    y = pm.Normal("y", mu=exp_trend, sigma=sigma, observed=demand)

    # Execute Inference
    trace_reg = pm.sample(draws=2000, chains=2, cores=2, target_accept=0.95, progressbar=False, random_seed=42)

    # Forensic Mandate: Always interrogate the unconstrained system
    prior_reg = pm.sample_prior_predictive(draws=100, random_seed=42)
    posterior_reg = pm.sample_posterior_predictive(trace_reg, random_seed=42, progressbar=False)

In [ ]:

fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: The Prior
# Extract the first chain of prior deterministic lines
prior_lines = prior_reg.prior["exp_trend"].isel(chain=0).values

ax[0].plot(t_months, demand, "k.", label="Demand Target")
for i in range(100):  # Plot 100 possible straight lines the model believed BEFORE data
    ax[0].plot(t_months, prior_lines[i], color="red", alpha=0.05)

ax[0].set_ylim(-50, 100)
ax[0].set_title("Prior Forensics: Spaghetti Lines of Possible Economies")
ax[0].legend()

# Plot 2: Posterior failure of the rigid Linear Regression
post_reg_trend = trace_reg.posterior["exp_trend"].mean(dim=["chain", "draw"])
hdi_reg = az.hdi(trace_reg.posterior["exp_trend"], hdi_prob=0.95)["exp_trend"]

ax[1].plot(t_months, demand, "k.", label="Raw Demand Data (Boom & Crash)")
ax[1].plot(t_months, post_reg_trend, "r-", label="Rigid Linear Regression", linewidth=2)
ax[1].fill_between(t_months, hdi_reg[:, 0], hdi_reg[:, 1], color="red", alpha=0.2, label="95% HDI")

ax[1].legend()
ax[1].set_title("Posterior Inference: Linear Regression Missing the Black Swan")

plt.tight_layout()
plt.show()

In [ ]:
# Re-deploying GRW on the Black Swan data
with pm.Model() as bsts_trend_2:
    sigma_trend = pm.HalfNormal("sigma_trend", sigma=0.5)
    # k>0.7 signifying overfit, changed sigma from 2 to 0.5 allowing smaller jummps and changing
    # to a number diff than the noise itself
    trend_component = pm.GaussianRandomWalk("trend_component", sigma=sigma_trend, shape=36)

    sigma_obs = pm.HalfNormal("sigma_obs", sigma=5)
    y = pm.Normal("y", mu=trend_component, sigma=sigma_obs, observed=demand)

    trace_trend_2 = pm.sample(draws=2000, chains=2, cores=2, target_accept=0.95, progressbar=False, random_seed=42)


post_trend_2 = trace_trend_2.posterior["trend_component"].mean(dim=["chain", "draw"])
hdi_2 = az.hdi(trace_trend_2.posterior["trend_component"], hdi_prob=0.95)["trend_component"]

plt.figure(figsize=(10, 5))
plt.plot(t_months, demand, "k.", label="Raw Demand Data (Boom & Crash)")
plt.plot(t_months, post_trend_2, "r-", label="BSTS Recovered Trend (GRW)", linewidth=2)
plt.fill_between(t_months, hdi_2[:, 0], hdi_2[:, 1], color="red", alpha=0.2, label="95% HDI")

plt.legend()
plt.title("Commercial Security: GRW Successfully Tracking the Structural Break")
plt.tight_layout()
plt.show()

In [ ]:
# Auditing for Overfitting
# A GRW has 36 hidden parameters (one for each time step). This is massive flexibility.
# use loo-cv

# Compute log likelihoods required for ArviZ comparisons
with reg_model:
    pm.compute_log_likelihood(trace_reg)
with bsts_trend_2:
    pm.compute_log_likelihood(trace_trend_2)

# 1. Pareto k Diagnostic: Check if specific data points aggressively dictate the fit
loo_grw = az.loo(trace_trend_2)
az.plot_khat(loo_grw, show_bins=True)
plt.axhline(0.7, color='red', linestyle='--')
plt.title("Forensic Audit: Pareto k Diagnostic (Values > 0.7 = Overfitting Warning)")
plt.tight_layout()
plt.show()

# 2. Information Criteria Comparison: Penalizing the model for its complexity
comp = az.compare({"Rigid LR": trace_reg, "GRW": trace_trend_2})
print(comp)

az.plot_compare(comp, insample_dev=False)
plt.title("Model Supremacy: LOO-CV Comparison (Penalized for Complexity)")
plt.tight_layout()
plt.show()

In [ ]:
# work remains
# since k>0.7, either observed normal change or use student t random walk

with pm.Model() as bsts_heavy_tail:

    sigma_trend = pm.HalfNormal("sigma_trend", sigma=2)
    nu = pm.Exponential("nu", lam=1/30)
    innovations = pm.StudentT("innovations", nu=nu, mu=0, sigma=sigma_trend, shape=36)
    # "innovations" are the independent shocks (or steps) that hit the system at each time period.

    # We sample the CHANGE from the distribution, not the absolute magnitude like we did beofre. vt-1+ delta v (change) = vt

    trend_component = pm.Deterministic("trend_component", innovations.cumsum())

    # when we do trend comp on gaussian walk it auto gens cumsum under the hood
    # but no direct exists so we do this

    sigma_obs = pm.HalfNormal("sigma_obs", sigma=2)
    # reduce from 5 to 2 or else higher jump values delegated as measurement errors rather than truth
    y = pm.Normal("y", mu=trend_component, sigma=sigma_obs, observed=demand)

    # In massive N=36 datasets with extreme shocks, the geometry is incredibly harsh.
    # Increase the tuning steps significantly so the sampler can map the steep cliffs of the Black Swan.
    trace_heavy_tail = pm.sample(draws=3000, tune=3000, chains=4, cores=4, target_accept=0.99, progressbar=False, random_seed=42)
    pm.compute_log_likelihood(trace_heavy_tail)

loo_srw = az.loo(trace_heavy_tail)
az.plot_khat(loo_srw, show_bins=True)
plt.title("Forensic Audit: SRW Pareto k Diagnostic (Checking Fix)")
plt.show()

In [ ]:
print(f"Max Pareto k (Gaussian RW): {loo_grw.pareto_k.max():.3f}")
print(f"Max Pareto k (Student-T RW): {loo_srw.pareto_k.max():.3f}")

# we have k values for every single data point
# loo cv, drop 1 can i get similar model without the same data point?
# or does the entire model collapse?

- still i got high studenT rw = 0.747 from 0.8 not much improvement (gaussian and studentT comp)

- the error might be our jump in values in months is high but only 0.5 stdev, even with fat tails extremely hight stdev is highly unlikely
- so change from `innovations = pm.StudentT("innovations", nu=nu, mu=0, sigma=0.5, shape=36)` to ...sigma = prior_distro
- Never hardcode a risk parameter when the data has the power to teach you the baseline truth. The model must learn its own volatility.
- we are not god, but we can at least tune our priors properly according to the data (no NOT on data but only physical possibilities, allowing model to find the actual underlying value for itself) we have rather than playing extremely dumb or taking the prior setting move for granted
- We use `HalfNormal(sigma=2)` not to force the jump to be 2, but to allow the *engine to learn* any jump size, while mathematically advising it that a baseline monthly volatility of ~2 units is physically reasonable for this specific market.

- If we build a Heavy-Tailed engine to model a Black Swan, but leave the sensor noise loose (`sigma_obs = HalfNormal(5)`), the model gets lazy. It sees the +15 jump and thinks: *"The sensor is allowed to be wrong by huge margins anyway. This +15 is probably just gauge error, not a true economic Boom."* 
It refuses to track the Black Swan, causing the MCMC to fail its architecture audit. We cannot hunt a Black Swan with a blurry telescope.



In [ ]:
print(f"Max Pareto k (Gaussian RW): {loo_grw.pareto_k.max():.3f}")
print(f"Max Pareto k (Student-T RW): {loo_srw.pareto_k.max():.3f}")

# Which exact months are blowing up the architecture?
failing_months = np.where(loo_srw.pareto_k > 0.7)[0]
print(f"\nMonths with k > 0.7 (Structural Failure Points): {failing_months}")
print(f"Total number of failing months: {len(failing_months)}")

# Print the specific k-values for the failing months to see the magnitude of the failure
for month in failing_months:
    print(f"Month {month}: k = {loo_srw.pareto_k[month]:.3f}")